In [48]:
import os
from pathlib import *
import subprocess
import shutil

In [49]:
contracts_dir = Path('../contracts')
cn_folders = list(os.listdir(contracts_dir))
cn_folders

['3']

In [50]:
curr_folder = cn_folders[0]
src_contracts = Path.joinpath(contracts_dir,curr_folder)
src_contracts

PosixPath('../contracts/3')

In [ ]:
#copiar tudo pra pasta do hardhat/contracts pra parar de dar erro nos imports
dest_contracts = Path('../hhat/contracts').resolve()
#print(dest_contracts)
dest_contracts.mkdir(parents=True,exist_ok=True)
for sol_file in src_contracts.rglob("*.sol"):
    shutil.copy(sol_file,dest_contracts)

/home/balantedaboavista/Documents/projects/ic/AuditAgents/hhat/contracts


In [55]:
import re
import subprocess
from pathlib import Path

# Set directories relative to src/
CONTRACTS_DIR = Path("../contracts/3").resolve()
HHAT_DIR = Path("../hhat").resolve()

def extract_npm_packages(sol_dir):
    """Scans all .sol files and extracts root npm package names from import statements."""
    packages = set()
    # Matches import statements like: import "@openzeppelin/contracts/token/ERC20/ERC20.sol";
    import_pattern = re.compile(r'import\s+["\'](@?[^"\';]+)["\'];')

    for sol_file in sol_dir.glob("**/*.sol"):
        with open(sol_file, "r", encoding="utf-8") as f:
            content = f.read()
            matches = import_pattern.findall(content)
            for path in matches:
                # Ignore relative local imports (e.g., "./Foo.sol" or "../Bar.sol")
                if path.startswith("."):
                    continue
                
                # Handle scoped packages (@openzeppelin/contracts) vs non-scoped (hardhat/console.sol)
                parts = path.split("/")
                if path.startswith("@"):
                    if len(parts) >= 2:
                        package_name = f"{parts[0]}/{parts[1]}"
                        packages.add(package_name)
                else:
                    packages.add(parts[0])

    return list(packages)

# Step 1: Detect required dependencies
packages_to_install = extract_npm_packages(CONTRACTS_DIR)
print(f"Detected dependencies: {packages_to_install}")

# Step 2: Install detected dependencies in the hhat folder
if packages_to_install:
    install_cmd = ["npm", "install", "--save-dev"] + packages_to_install
    print(f"Installing dependencies in {HHAT_DIR}...")
    
    install_result = subprocess.run(
        install_cmd,
        cwd=HHAT_DIR,
        capture_output=True,
        text=True
    )
    
    if install_result.returncode == 0:
        print("Dependencies installed successfully!")
    else:
        print("Failed to install dependencies:")
        print(install_result.stderr)


Detected dependencies: ['@uniswap/v2-core', '@openzeppelin/contracts']
Installing dependencies in /home/balantedaboavista/Documents/projects/ic/AuditAgents/hhat...
Dependencies installed successfully!


In [56]:
result = subprocess.run(
    ["npx", "hardhat", "compile"],
    cwd=Path("../hhat").resolve(),
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else result.stderr)

Error HHE902: There was an error while resolving the import "../libraries/UniswapStyleLib.sol" from "./contracts/PriceAware.sol":

The file "libraries/UniswapStyleLib.sol" doesn't exist within the project.

For more info go to https://hardhat.org/HHE902 or run Hardhat with --show-stack-traces

